# Multi-Cloud Notebook Authenticator & Safe Teardown

This notebook demonstrates a reusable provider-authentication interface for GCP, AWS, Azure, and Oracle Cloud. GCP is the primary path. All teardown is preview-first and limited to resources explicitly registered by this session.

## 1. Install dependencies

Run only the providers you plan to use. In Colab, the package folder must be uploaded or mounted first.

In [ ]:
%pip install "google-auth>=2.29"
# Optional providers:
# %pip install "boto3>=1.34" "azure-identity>=1.16" "oci>=2.120"

## 2. Import the shared SDK

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys

# Update this path if you chose a different Drive folder name.
SDK_PATH = "/content/drive/MyDrive/Colab_Notebook_Library_2026-11-06"
sys.path.insert(0, SDK_PATH)
from notebook_cloud import CloudSession


## 3. Authenticate to GCP (default path)

Colab user authentication uses Application Default Credentials. For service-account impersonation, provide the target service account. For WIF, configure `GOOGLE_APPLICATION_CREDENTIALS` to point at the external-account credential configuration.

In [ ]:
PROJECT_ID = "YOUR_GCP_PROJECT_ID"  # replace before running
cloud = CloudSession("gcp", project_id=PROJECT_ID)
auth = cloud.authenticate()
print({"provider": auth.provider, "project": auth.project_or_account, "mode": auth.details})

## 4. Other provider examples

Uncomment only the provider you intend to authenticate. Each uses its standard credential chain or native profile.

In [ ]:
# AWS: boto3 default credential chain; optionally pass role_arn="arn:aws:iam::...:role/..."
# aws = CloudSession("aws", region="us-east-1").authenticate()

# Azure: DefaultAzureCredential; set interactive=True only when browser login is intended
# azure = CloudSession("azure", tenant_id="YOUR_TENANT_ID").authenticate()

# OCI: use ~/.oci/config profile locally, or instance principals on OCI compute
# oci = CloudSession("oci", profile="DEFAULT").authenticate()

## 5. Register resources created by this notebook

Register each created resource immediately after successful creation. Supply a delete handler that targets only that exact resource.

In [ ]:
# Example only: after creating a bucket through the GCP SDK, register its exact name.
# bucket = storage_client.bucket("my-notebook-created-bucket")
# cloud.register_resource("gcs_bucket", bucket.name, delete_action=lambda: bucket.delete(force=True))

# Inspect what this session is allowed to clean up.
cloud.plan_destroy()

## 6. Preview, then explicitly confirm teardown

The default call is a dry-run preview. No deletion occurs unless `confirm=True` and `dry_run=False` are both provided. Resources without a registered delete handler are skipped.

In [ ]:
# Preview only
preview = cloud.destroy()
preview

# Execute only after reviewing the preview and confirming resource ownership/data impact:
# result = cloud.destroy(confirm=True, dry_run=False)
# result

## 7. Notebook-end cleanup hook

Keep this as a preview unless you deliberately opt into deletion. Do not run destructive cleanup automatically in a notebook that may be interrupted or rerun.

In [ ]:
# Safe default at notebook end
cloud.cleanup()

# Explicit destructive cleanup (uncomment only after reviewing the plan):
# cloud.cleanup(confirm=True, dry_run=False)

## Security notes

- Never paste service-account JSON keys, AWS secret keys, Azure client secrets, or OCI private keys into notebook cells.
- Prefer short-lived credentials, ADC, WIF, managed identities, and role impersonation.
- Keep cloud permissions least-privilege.
- The session registry is not a substitute for cloud billing alerts, resource policies, or Terraform state.
- Only resources explicitly registered by this session are eligible for teardown.